### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 

- Tracking agent behaviour with logging, analytics, and debugging.
- Transforming prompts, tool selection , and output formatting.
- Adding retries ,fallbacks, and early termination logic.
- Applying rate limits, guardrails, P detection

Summarization Middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
import os


model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)


message length

In [ ]:
agent = create_agent(
    model = model,
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 3)
        )
    ]  
)

# run with thread
config = {"configurable": {"thread_id" : "test-1"}}

In [21]:
questions= [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4+13?",
    "what is 45/9?"
]

for q in questions:
    response = agent.invoke({"messages" : [HumanMessage(content=q)]}, config)
    print(f"messages: {response}")
    print(f"message_length = {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='90bd5842-4055-4295-9385-5a2083a074be'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EnMKcQFpFH0TSTill5tYaEENQSKu4TM9koxBZgvgBcilNqG7lOuCBQz9sPradHKoyOkyiK9dt1UPsvR+qM42LDzhJasgYSDV0Q7kVk0x4ZIrno2LT1GG/nU/+km0jbUIY7REldM/1ZAKwYs942lhgk5XBCNB'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a15233-becc-7262-9059-b44c3915f97a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
message_length = 2
messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='90bd5842-4055-4295-9385-5a2083a074be'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras':

token size

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
import os


@tool
def search_hotel(city: str)-> str:
    ''' Search hotels - return long response to use more tokens.'''
    return f"""Hotel in {city}:
    1. Marriott Hotel - 5 star, 1000$/ night, spa, pool, gym
    1. City Inn - 4 star, 100$/ night, business center
    1. Budget friendly - 3 star, 69$/ night, free wifi"""

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

agent = create_agent(
    model = model,
    checkpointer= InMemorySaver(),
    tools=[search_hotel],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 400),
            keep=("tokens", 200),
        )
    ]  
)

# run with thread
config = {"configurable": {"thread_id" : "test-1"}}

# '''to count the number of tokens used (approximate)'''
# def cnt_tokens(messages):
#     total_chars = sum(len(str(m.content)) for m in messages)
#     return total_chars // 4 # 4 chars = 1 token

# '''to count the number of tokens used (approximate)'''
def cnt_tokens(messages):
    return model.get_num_tokens_from_messages(messages)

In [19]:
# run test
cities = ["Delhi", "Kolkata", "Mumbai", "Chennai", "Pune", "Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content= f"find hotels in {city}")]},
        config= config
    )

    tokens = cnt_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    # print(f"{(response['messages'])}")
    print(response["messages"][-1].content)

Delhi: ~217 tokens, 4 messages
[{'type': 'text', 'text': 'Here are a few hotel options in Delhi:\n\n*   **Marriott Hotel:** A 5-star property offering luxury amenities including a spa, pool, and gym. Price: $1000 per night.\n*   **City Inn:** A 4-star hotel that features a business center. Price: $100 per night.\n*   **Budget Friendly:** A 3-star hotel providing free Wi-Fi. Price: $69 per night.', 'extras': {'signature': 'EnMKcQFpFH0TdzHfc8c55bVNwi2GrkDwAIoTgpyABCiPWsp3gtFuqQZ6fsZ9t0acoabsjmMRoFzka9Qpm2VDRjjnIUXVibXpItdv1n66pZsSIy+BIaRZl8ZrGSiA4cbeH0Rz2OSUt9NjaOiwjA2Q2GbhU/Rn'}}]
Kolkata: ~432 tokens, 8 messages
[{'type': 'text', 'text': 'Here are some hotel options in Kolkata:\n\n*   **Marriott Hotel:** A 5-star hotel featuring a spa, pool, and gym. Price: $1000 per night.\n*   **City Inn:** A 4-star hotel equipped with a business center. Price: $100 per night.\n*   **Budget Friendly:** A 3-star hotel offering free Wi-Fi. Price: $69 per night.', 'extras': {'signature': 'EnMKcQFpFH0TKC

fraction

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~109 tokens (0.0852%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='33455a29-a18e-45b6-adaa-f61f7aa1684d'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_543855': 'EnMKcQFpFH0TNgT+riZwhZ+FbtOk74SVov0oSkWSIHwJ9bPh3VXyyYQvsdsWb/yrwy7qK3685q6QbkW8CsTi4YDJLehLnwuMBUq3qgKeEnmRCgYb+hS2iKV/GDQ2WYVfZ3GedDpJnPrVskZSETWcK4MkHNd+'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a15850-8f96-75c2-85c2-81d3b15156c9-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_543855', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris: Grand 

##### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

import os
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False, # don't need any human approval 
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [3]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ad34d435-b790-4354-ab50-ad5df8398aff'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_211323': 'EnMKcQFpFH0Ty5ai0K2r8gGBMK3iE+lv49sSCSR4ytFvcXv4cSzaSq0NDY7kIc56LqjKFUgiitPweC81YXzzAktg2UBCur0Q6ROTYOaeDmg9Q2Uk8yPg81ZwVgNxsf1d/1TyHgfuKm8umtsGhRw0EjPOrfaQ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a159d4-6e40-7261-8c56-8c3225e7df8b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': 'call_211323', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

Approve

In [4]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EnMKcQFpFH0TMRzYp7Vx0Ba1KCY4TUlreNcmNeoVLuUbqvihgfkA70ARAkBR+MJZKDJiE0xLbQsfiRWT2DBRA24nNzZDMrsRpJEpbm+JfCPaqyoG/1JhBD54z9WcIhvZXFhTt9yuVUIdOxsRNTjwUgkn9n2C'}}]


In [5]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ad34d435-b790-4354-ab50-ad5df8398aff'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_211323': 'EnMKcQFpFH0Ty5ai0K2r8gGBMK3iE+lv49sSCSR4ytFvcXv4cSzaSq0NDY7kIc56LqjKFUgiitPweC81YXzzAktg2UBCur0Q6ROTYOaeDmg9Q2Uk8yPg81ZwVgNxsf1d/1TyHgfuKm8umtsGhRw0EjPOrfaQ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a159d4-6e40-7261-8c56-8c3225e7df8b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': 'call_211323', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

Reject

In [9]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].text}")

⏸️ Paused! Approving...
✅ Result: The email was not sent because the action was rejected. If you would like me to proceed with sending the email, please let me know.


In [3]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='6ec69e8c-eb3e-4def-b40b-6b00e9e07daf'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_613220': 'EnMKcQFpFH0TB0kYaLSXkSJsicHJhihWS3WH9N0vcwGvoz94UQJzqnUEQlApbuCu91kOgI6Q4Xr1aOmUoJ/yiQzqsDaXYNtLK3/8Y8uxAh8VU74xXXxjDvcQolkaZj1NTf9OXL/sc93/c8pHO0OL1cvMZZSc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a15c17-cd2d-7eb2-bffe-195714e7f866-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_613220', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

Edit

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

import os
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False, # don't need any human approval 
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [13]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='69d56fa3-446d-4365-bcde-244226e0179c'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_750596': 'EnMKcQFpFH0TCz9HHuwRGKdWLH0GW2HQzBe2BpPb+mWhmRFWyOIaUBKaiETITHK34WuHjlPpZFBoRnBrMCYxtd+PhqMNEOmhe1ZWYEDY6ri8CecukOUqsmKNNdSNM9Drs3WSHMsjYw2UwKlMgR2juHW3PD3m'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a15c1f-e5b0-7473-84bf-020f1ca1d820-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': 'call_750596', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [20]:

# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-2].content}")

⏸️ Paused! Editing...
✏️ Result: Email sent to correct@email.com with subject 'Corrected Subject'


In [15]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='69d56fa3-446d-4365-bcde-244226e0179c'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_750596': 'EnMKcQFpFH0TCz9HHuwRGKdWLH0GW2HQzBe2BpPb+mWhmRFWyOIaUBKaiETITHK34WuHjlPpZFBoRnBrMCYxtd+PhqMNEOmhe1ZWYEDY6ri8CecukOUqsmKNNdSNM9Drs3WSHMsjYw2UwKlMgR2juHW3PD3m'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a15c1f-e5b0-7473-84bf-020f1ca1d820-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'call_750